# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [2]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1922


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [3]:
# YOUR CHANGES HERE

# def random_policy(state, actions):
#     return 0

import random
import pandas as pd

def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [4]:
# YOUR CHANGES HERE

#for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
#    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

rows = []

for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
    rows.append({
        "curr_state": curr_state,
        "curr_action": curr_action,
        "next_reward": next_reward,
        "next_state": next_state,
    })

pd.DataFrame(rows).to_csv("log-random.tsv", sep="\t", index=False)

CURR STATE 1091 ACTION 0 NEXT REWARD 0 NEXT STATE 971
CURR STATE 971 ACTION -1 NEXT REWARD 0 NEXT STATE 850
CURR STATE 850 ACTION 0 NEXT REWARD 0 NEXT STATE 858
CURR STATE 858 ACTION 1 NEXT REWARD 0 NEXT STATE 867
CURR STATE 867 ACTION -1 NEXT REWARD 0 NEXT STATE 746
CURR STATE 746 ACTION 1 NEXT REWARD 0 NEXT STATE 755
CURR STATE 755 ACTION 0 NEXT REWARD 0 NEXT STATE 635
CURR STATE 635 ACTION -1 NEXT REWARD 0 NEXT STATE 506
CURR STATE 506 ACTION -1 NEXT REWARD 0 NEXT STATE 505
CURR STATE 505 ACTION 0 NEXT REWARD 0 NEXT STATE 633
CURR STATE 633 ACTION 1 NEXT REWARD 0 NEXT STATE 762
CURR STATE 762 ACTION -1 NEXT REWARD 0 NEXT STATE 761
CURR STATE 761 ACTION -1 NEXT REWARD 0 NEXT STATE 888
CURR STATE 888 ACTION 1 NEXT REWARD 0 NEXT STATE 1017
CURR STATE 1017 ACTION 0 NEXT REWARD 0 NEXT STATE 1145
CURR STATE 1145 ACTION 0 NEXT REWARD 0 NEXT STATE 1273
CURR STATE 1273 ACTION 1 NEXT REWARD 0 NEXT STATE 1402
CURR STATE 1402 ACTION 0 NEXT REWARD 0 NEXT STATE 1402
CURR STATE 1402 ACTION 0 NEXT 

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [5]:
# YOUR CHANGES HERE

import random
import pandas as pd

def random_policy(state, actions):
    return random.choice(actions)

alpha = 0.5
gamma = 0.5

q_values = {}
rows = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        old_value = q_values.get((curr_state, curr_action), 0.0)
        next_actions = simulator.get_actions(next_state)
        next_max = max(q_values.get((next_state, a), 0.0) for a in next_actions)

        new_value = old_value + alpha * (next_reward + gamma * next_max - old_value)
        q_values[(curr_state, curr_action)] = new_value

        rows.append({
            "curr_state": curr_state,
            "curr_action": curr_action,
            "next_reward": next_reward,
            "next_state": next_state,
            "old_value": old_value,
            "new_value": new_value,
        })

Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [6]:
# YOUR CHANGES HERE


pd.DataFrame(rows).to_csv("q-random.tsv", sep="\t", index=False)

Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [7]:
# YOUR CHANGES HERE

# hard-code epsilon=0.25. this is high but the environment is deterministic.
#def epsilon_greedy_policy(state, actions):
 #   return 0

alpha = 0.5
gamma = 0.5

q_values = {}

def epsilon_greedy_policy(state, actions):
    if random.random() < 0.25:
        return random.choice(actions)

    best_value = max(q_values.get((state, a), 0.0) for a in actions)
    best_actions = [a for a in actions if q_values.get((state, a), 0.0) == best_value]
    return random.choice(best_actions)

rows = []



Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [8]:
# YOUR CHANGES HERE
for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        old_value = q_values.get((curr_state, curr_action), 0.0)
        next_actions = simulator.get_actions(next_state)
        next_max = max(q_values.get((next_state, a), 0.0) for a in next_actions)

        new_value = old_value + alpha * (next_reward + gamma * next_max - old_value)
        q_values[(curr_state, curr_action)] = new_value

        rows.append({
            "curr_state": curr_state,
            "curr_action": curr_action,
            "next_reward": next_reward,
            "next_state": next_state,
            "old_value": old_value,
            "new_value": new_value,
        })

pd.DataFrame(rows).to_csv("q-greedy.tsv", sep="\t", index=False)

Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [9]:
# YOUR CHANGES HERE

qg = pd.read_csv("q-greedy.tsv", sep="\t")
states = sorted(qg["curr_state"].unique())

rows = []
for state in states:
    actions = simulator.get_actions(state)
    best_value = max(q_values.get((state, a), 0.0) for a in actions)
    best_actions = [a for a in actions if q_values.get((state, a), 0.0) == best_value]
    rows.append({"state": state, "action": best_actions[0]})

pd.DataFrame(rows).to_csv("policy-greedy.tsv", sep="\t", index=False)

Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [10]:
# YOUR CHANGES HERE
from collections import deque, defaultdict

num_states = simulator.resolution * simulator.resolution * 8
actions = [-1, 0, 1]
terminal = simulator.terminal_state

# Build reverse graph and shortest distances to terminal
predecessors = defaultdict(list)
distance = {terminal: 0}

for state in range(num_states):
    for action in actions:
        reward, next_state = simulator.get_next_reward_state(state, action)
        predecessors[next_state].append((state, action))

queue = deque([terminal])

while queue:
    s = queue.popleft()
    for prev_state, prev_action in predecessors[s]:
        if prev_state not in distance:
            distance[prev_state] = distance[s] + 1
            queue.append(prev_state)

# Extract a no-loop shortest-path policy for all states
rows = []

for state in range(num_states):
    if state == terminal:
        action = 0
    else:
        best_action = None
        best_dist = float("inf")

        for action_candidate in actions:
            reward, next_state = simulator.get_next_reward_state(state, action_candidate)

            if reward > 0:
                best_action = action_candidate
                best_dist = -1
                break

            d = distance.get(next_state, float("inf"))
            if d < best_dist:
                best_dist = d
                best_action = action_candidate

        if best_action is None:
            best_action = 0

        action = best_action

    rows.append({
        "state": state,
        "action": action
    })

pd.DataFrame(rows).to_csv("policy-optimal.tsv", sep="\t", index=False)

Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.